In [1]:
import os
%pwd

'/home/tuhin/bangla-political-memes-classification/research'

In [2]:
import os
if os.path.basename(os.getcwd()) == 'research':
    os.chdir("../")

In [3]:
%pwd

'/home/tuhin/bangla-political-memes-classification'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ClassificationModelUsingTextConfig:
    root_dir: Path
    train_features_csv: Path
    test_features_csv: Path
    model_path: Path

In [5]:
from memeClassifier.constants import *
from memeClassifier.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_classification_model_using_text_config(self) -> ClassificationModelUsingTextConfig:
        config = self.config.classification_model_using_text

        create_directories([config.root_dir])

        classification_config = ClassificationModelUsingTextConfig(
            root_dir=Path(config.root_dir),
            train_features_csv=Path(config.train_features_csv),
            test_features_csv=Path(config.test_features_csv),
            model_path=Path(config.model_path)
        )

        return classification_config

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib
from memeClassifier import logger

class ClassificationModelUsingText:
    def __init__(self, config: ClassificationModelUsingTextConfig):
        self.config = config

    def initiate_model_training(self):
        logger.info("Loading feature datasets")
        train_df = pd.read_csv(self.config.train_features_csv)
        
        X = train_df[['political_specific_count', 'political_specific_ratio']]
        y = train_df['Label'].map({'Political': 1, 'NonPolitical': 0})
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        logger.info(f"Training set size: {X_train.shape[0]}")
        logger.info(f"Validation set size: {X_val.shape[0]}")
        
        model = LogisticRegression(random_state=42, max_iter=1000)
        model.fit(X_train, y_train)
        
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        
        train_accuracy = accuracy_score(y_train, y_train_pred)
        val_accuracy = accuracy_score(y_val, y_val_pred)
        
        logger.info(f"Training Accuracy: {train_accuracy:.4f}")
        logger.info(f"Validation Accuracy: {val_accuracy:.4f}")
        logger.info(f"Classification Report (Validation Set):\n{classification_report(y_val, y_val_pred, target_names=['NonPolitical', 'Political'])}")
        
        joblib.dump(model, self.config.model_path)
        logger.info(f"Model saved to {self.config.model_path}")

In [8]:
try:
    config = ConfigurationManager()
    classification_config = config.get_classification_model_using_text_config()
    classifier = ClassificationModelUsingText(config=classification_config)
    classifier.initiate_model_training()
except Exception as e:
    raise e

[2026-06-26 19:11:22,266: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-06-26 19:11:22,268: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-26 19:11:22,269: INFO: common: Directory created at: artifacts]
[2026-06-26 19:11:22,270: INFO: common: Directory created at: artifacts/classification_model_using_text]
[2026-06-26 19:11:22,270: INFO: 214661114: Loading feature datasets]
[2026-06-26 19:11:22,278: INFO: 214661114: Training set size: 156]
[2026-06-26 19:11:22,279: INFO: 214661114: Validation set size: 39]
[2026-06-26 19:11:22,290: INFO: 214661114: Training Accuracy: 0.7436]
[2026-06-26 19:11:22,291: INFO: 214661114: Validation Accuracy: 0.7179]
[2026-06-26 19:11:22,297: INFO: 214661114: Classification Report (Validation Set):
              precision    recall  f1-score   support

NonPolitical       0.75      0.93      0.83        29
   Political       0.33      0.10      0.15        10

    accuracy                           0.72        39
